# Quick-Commerce Financial Analysis
### Comparing DoorDash, Delivery Hero, and Jumia — 2022 to 2024

**Objective:** Identify which company represents the best investment opportunity in Quick Commerce based on revenue growth, profitability trajectory, and cash position.

**Companies:** DoorDash (DASH) · Delivery Hero (DHER.DE) · Jumia (JMIA)  
**Period:** 2022 — 2023 — 2024  
**Tools:** Python · pandas · yfinance · Power BI

---

In [1]:
import pandas as pd
import numpy as np
import yfinance as yf

---
## Phase 1 — Data Collection

Financial statements pulled directly from Yahoo Finance via `yfinance`.
Each company object contains the income statement, balance sheet, and cash flow statement.
Unit economics (GMV, Adjusted EBITDA) are sourced manually from annual reports in Phase 2.

In [2]:
tickers = {
    'DoorDash':      'DASH',
    'Delivery Hero': 'DHER.DE',
    'Jumia':         'JMIA'
}

data = {}
for name, ticker in tickers.items():
    company = yf.Ticker(ticker)
    data[name] = {
        'income_statement': company.financials,
        'balance_sheet':    company.balance_sheet,
        'cash_flow':        company.cashflow
    }
    print(f"{name} — done")

DoorDash — done
Delivery Hero — done
Jumia — done


---
## Phase 2 — Data Cleaning, Standardization & Derived Metrics

**Steps in this phase:**
1. Apply EUR → USD FX conversion for Delivery Hero
2. Extract all financial metrics from yfinance data objects
3. Merge manual inputs (GMV, Adjusted EBITDA) from annual reports
4. Calculate six derived metrics per company per year
5. Build the master DataFrame (9 rows × 17 columns)
6. Export to Excel

In [3]:
# Average annual EUR/USD exchange rates — source: ECB / Federal Reserve
# Delivery Hero reports in EUR; all figures converted to USD for comparability
fx_rates = {
    '2022-12-31': 1.053,
    '2023-12-31': 1.082,
    '2024-12-31': 1.085
}

target_years = ['2022-12-31', '2023-12-31', '2024-12-31']

In [4]:
def extract_row(company_name, statement, metric, cols):
    """Extract a single metric row from a yfinance statement."""
    df = financial_data[company_name][statement]
    available_cols = [c for c in cols if c in df.columns]
    if metric in df.index:
        return df.loc[metric, available_cols]
    return pd.Series([None] * len(available_cols), index=available_cols)

# Rename to avoid overwriting the Phase 1 data dict
financial_data = data.copy()

company_currency = {
    'DoorDash':      'USD',
    'Delivery Hero': 'EUR',
    'Jumia':         'USD'
}

rows = []

for company in ['DoorDash', 'Delivery Hero', 'Jumia']:
    currency = company_currency[company]

    for year_col in target_years:
        year = int(year_col[:4])
        fx   = fx_rates[year_col] if currency == 'EUR' else 1.0

        def get(statement, metric):
            series = extract_row(company, statement, metric, [year_col])
            val    = series.get(year_col, None)
            if val is None or pd.isna(val):
                return None
            return round((val / 1e6) * fx, 1)  # Convert to millions USD

        rows.append({
            'Company':              company,
            'Year':                 year,
            'Revenue_USD':          get('income_statement', 'Total Revenue'),
            'Gross_Profit_USD':     get('income_statement', 'Gross Profit'),
            'Operating_Income_USD': get('income_statement', 'Operating Income'),
            'EBITDA_USD':           get('income_statement', 'EBITDA'),
            'Net_Income_USD':       get('income_statement', 'Net Income'),
            'Cash_USD':             get('balance_sheet',    'Cash And Cash Equivalents'),
            'Operating_CF_USD':     get('cash_flow',        'Operating Cash Flow'),
        })

df = pd.DataFrame(rows)

print("Raw extraction complete — FX conversion applied")
print(df[['Company', 'Year', 'Revenue_USD', 'Gross_Profit_USD']].to_string(index=False))

Raw extraction complete — FX conversion applied
      Company  Year  Revenue_USD  Gross_Profit_USD
     DoorDash  2022       6583.0            2995.0
     DoorDash  2023       8635.0            4046.0
     DoorDash  2024      10722.0            5180.0
Delivery Hero  2022       9032.0            2350.1
Delivery Hero  2023      10757.0            3216.5
Delivery Hero  2024      13339.9            3612.2
        Jumia  2022        203.3             118.2
        Jumia  2023        186.4             107.1
        Jumia  2024        167.5              99.5


In [5]:
# GMV and Adjusted EBITDA are not available in yfinance — sourced from annual reports:
# DoorDash 10-K | Delivery Hero Annual Report | Jumia 20-F
# Delivery Hero figures in EUR are converted to USD using the same annual FX rates

manual = {
    ('DoorDash',      2022): {'GMV_USD': 53414,  'Adj_EBITDA_USD': 361},
    ('DoorDash',      2023): {'GMV_USD': 66771,  'Adj_EBITDA_USD': 1190},
    ('DoorDash',      2024): {'GMV_USD': 80231,  'Adj_EBITDA_USD': 1900},
    ('Delivery Hero', 2022): {'GMV_USD': round(42827 * 1.053, 1), 'Adj_EBITDA_USD': round(-467.2 * 1.053, 1)},
    ('Delivery Hero', 2023): {'GMV_USD': round(45275 * 1.082, 1), 'Adj_EBITDA_USD': round(253.6  * 1.082, 1)},
    ('Delivery Hero', 2024): {'GMV_USD': round(48754 * 1.085, 1), 'Adj_EBITDA_USD': round(692.5  * 1.085, 1)},
    ('Jumia',         2022): {'GMV_USD': 932.5,  'Adj_EBITDA_USD': -182.1},
    ('Jumia',         2023): {'GMV_USD': 749.8,  'Adj_EBITDA_USD': -58.2},
    ('Jumia',         2024): {'GMV_USD': 720.6,  'Adj_EBITDA_USD': -51.3},
}

df['GMV_USD']        = df.apply(lambda r: manual[(r['Company'], r['Year'])]['GMV_USD'],        axis=1)
df['Adj_EBITDA_USD'] = df.apply(lambda r: manual[(r['Company'], r['Year'])]['Adj_EBITDA_USD'], axis=1)

print("Manual inputs merged — GMV and Adj. EBITDA from annual reports")

Manual inputs merged — GMV and Adj. EBITDA from annual reports


In [6]:
df = df.sort_values(['Company', 'Year']).reset_index(drop=True)

# Revenue growth — year-over-year percentage change per company
df['Revenue_Growth_Pct'] = (
    df.groupby('Company')['Revenue_USD'].pct_change() * 100
).round(1)

# Margin metrics
df['Gross_Margin_Pct']    = (df['Gross_Profit_USD']     / df['Revenue_USD'] * 100).round(1)
df['Operating_Margin_Pct']= (df['Operating_Income_USD'] / df['Revenue_USD'] * 100).round(1)
df['EBITDA_Margin_Pct']   = (df['Adj_EBITDA_USD']       / df['Revenue_USD'] * 100).round(1)
df['Net_Margin_Pct']      = (df['Net_Income_USD']        / df['Revenue_USD'] * 100).round(1)

# Unit economics efficiency — Adj. EBITDA as a percentage of GMV
df['EBITDA_over_GMV_Pct'] = (df['Adj_EBITDA_USD'] / df['GMV_USD'] * 100).round(2)

# Cash runway — only meaningful when Operating CF is negative
def cash_runway(row):
    if row['Operating_CF_USD'] is None or row['Operating_CF_USD'] >= 0:
        return None  # Cash-generative — runway metric not applicable
    monthly_burn = abs(row['Operating_CF_USD']) / 12
    return round(row['Cash_USD'] / monthly_burn, 1)

df['Cash_Runway_Months'] = df.apply(cash_runway, axis=1)

print("Derived metrics calculated")

Derived metrics calculated


In [7]:
master_df = df[[
    'Company', 'Year',
    'Revenue_USD', 'Revenue_Growth_Pct',
    'Gross_Profit_USD', 'Gross_Margin_Pct',
    'Operating_Income_USD', 'Operating_Margin_Pct',
    'Adj_EBITDA_USD', 'EBITDA_Margin_Pct',
    'Net_Income_USD', 'Net_Margin_Pct',
    'GMV_USD', 'EBITDA_over_GMV_Pct',
    'Cash_USD', 'Operating_CF_USD', 'Cash_Runway_Months'
]].reset_index(drop=True)

pd.options.display.float_format = '{:,.0f}'.format

print("Master DataFrame — 9 rows x 17 columns")
print(master_df.to_string(index=False))

Master DataFrame — 9 rows x 17 columns
      Company  Year  Revenue_USD  Revenue_Growth_Pct  Gross_Profit_USD  Gross_Margin_Pct  Operating_Income_USD  Operating_Margin_Pct  Adj_EBITDA_USD  EBITDA_Margin_Pct  Net_Income_USD  Net_Margin_Pct  GMV_USD  EBITDA_over_GMV_Pct  Cash_USD  Operating_CF_USD  Cash_Runway_Months
Delivery Hero  2022        9,032                 NaN             2,350                26                  -891                   -10            -492                 -5          -3,168             -35   45,097                   -1     2,546              -725                  42
Delivery Hero  2023       10,757                  19             3,216                30                  -198                    -2             274                  3          -2,486             -23   48,988                    1     1,796               -21               1,021
Delivery Hero  2024       13,340                  24             3,612                27                   478                 

In [8]:
excel_path = r"E:\ZiAd\Data Analysis Projects\Quick-Commerce Financial Analysis\DATA\qc_financial_data.xlsx"

with pd.ExcelWriter(excel_path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    master_df.to_excel(writer, sheet_name='Master Data', index=False)

print(f"Exported to: {excel_path}")

Exported to: E:\ZiAd\Data Analysis Projects\Quick-Commerce Financial Analysis\DATA\qc_financial_data.xlsx


### Methodology Notes

**Currency standardization:** Delivery Hero reports in EUR. All figures converted to USD using average annual EUR/USD rates (2022: 1.053 | 2023: 1.082 | 2024: 1.085). Average annual rates reflect real economic value per period rather than an arbitrary snapshot.

**Segment isolation:** Full quick-commerce segment isolation was not achievable for any of the three companies from public filings. DoorDash does not separate quick commerce. Delivery Hero does not publish Total Orders at group level. Jumia operates in e-commerce broadly. All analysis uses group-level figures as the best available public proxy — this is a data availability constraint, not an analytical choice.

**Cash runway:** Where Operating Cash Flow is positive, Cash Runway is marked `None` — the metric is only meaningful for cash-burning companies.

---
## Phase 3 — Analysis

Four analytical steps:
1. Revenue Growth
2. Profitability Trajectory
3. Cash Position
4. Investment Scorecard & Conclusion

### Step 1 — Revenue Growth

Measures absolute revenue scale, year-over-year growth rate, and 2-year CAGR per company.
The core question here: is the business growing, and is that growth accelerating or slowing?

In [9]:
print("--- Revenue (USD Millions) ---\n")

revenue_table = master_df[['Company', 'Year', 'Revenue_USD', 'Revenue_Growth_Pct']].copy()
revenue_table['Revenue_USD']        = revenue_table['Revenue_USD'].apply(lambda x: f"${x:,.0f}M")
revenue_table['Revenue_Growth_Pct'] = revenue_table['Revenue_Growth_Pct'].apply(
    lambda x: f"{x:+.1f}%" if pd.notna(x) else "—"
)

print(revenue_table.to_string(index=False))

--- Revenue (USD Millions) ---

      Company  Year Revenue_USD Revenue_Growth_Pct
Delivery Hero  2022     $9,032M                  —
Delivery Hero  2023    $10,757M             +19.1%
Delivery Hero  2024    $13,340M             +24.0%
     DoorDash  2022     $6,583M                  —
     DoorDash  2023     $8,635M             +31.2%
     DoorDash  2024    $10,722M             +24.2%
        Jumia  2022       $203M                  —
        Jumia  2023       $186M              -8.3%
        Jumia  2024       $168M             -10.1%


In [10]:
print("--- Revenue Growth Summary ---\n")

for company in ['DoorDash', 'Delivery Hero', 'Jumia']:
    subset   = master_df[master_df['Company'] == company].sort_values('Year')
    rev_2022 = subset[subset['Year'] == 2022]['Revenue_USD'].values[0]
    rev_2024 = subset[subset['Year'] == 2024]['Revenue_USD'].values[0]
    cagr     = ((rev_2024 / rev_2022) ** (1/2) - 1) * 100
    g_2023   = subset[subset['Year'] == 2023]['Revenue_Growth_Pct'].values[0]
    g_2024   = subset[subset['Year'] == 2024]['Revenue_Growth_Pct'].values[0]

    print(f"{company}")
    print(f"  Revenue 2022: ${rev_2022:,.0f}M  -->  2024: ${rev_2024:,.0f}M")
    print(f"  YoY Growth  : 2023 = {g_2023:+.1f}%  |  2024 = {g_2024:+.1f}%")
    print(f"  2-Year CAGR : {cagr:.1f}%")
    print()

--- Revenue Growth Summary ---

DoorDash
  Revenue 2022: $6,583M  -->  2024: $10,722M
  YoY Growth  : 2023 = +31.2%  |  2024 = +24.2%
  2-Year CAGR : 27.6%

Delivery Hero
  Revenue 2022: $9,032M  -->  2024: $13,340M
  YoY Growth  : 2023 = +19.1%  |  2024 = +24.0%
  2-Year CAGR : 21.5%

Jumia
  Revenue 2022: $203M  -->  2024: $168M
  YoY Growth  : 2023 = -8.3%  |  2024 = -10.1%
  2-Year CAGR : -9.2%



In [11]:
print("--- Analytical Observations ---\n")

observations = [
    "DoorDash: Consistent double-digit growth (+31% in 2023, +24% in 2024). "
    "Revenue scaled from $6.6B to $10.7B — a 28% CAGR over 2 years. "
    "Growth is organic, driven by GMV expansion, not acquisitions.",

    "Delivery Hero: Moderate growth (+19% in 2023, +24% in 2024) with a 21% CAGR. "
    "However, this growth is partly inflated by FX conversion (EUR reported). "
    "In EUR terms, growth is softer — 6% and 8% respectively.",

    "Jumia: Revenue is declining — -8% in 2023 and -10% in 2024. "
    "This is not a temporary dip — it reflects a shrinking customer base "
    "(7.3M active customers in 2022 down to 5.4M in 2024) "
    "and falling GMV ($932M to $721M). Not a growth story.",
]

for i, obs in enumerate(observations, 1):
    print(f"{i}. {obs}\n")

--- Analytical Observations ---

1. DoorDash: Consistent double-digit growth (+31% in 2023, +24% in 2024). Revenue scaled from $6.6B to $10.7B — a 28% CAGR over 2 years. Growth is organic, driven by GMV expansion, not acquisitions.

2. Delivery Hero: Moderate growth (+19% in 2023, +24% in 2024) with a 21% CAGR. However, this growth is partly inflated by FX conversion (EUR reported). In EUR terms, growth is softer — 6% and 8% respectively.

3. Jumia: Revenue is declining — -8% in 2023 and -10% in 2024. This is not a temporary dip — it reflects a shrinking customer base (7.3M active customers in 2022 down to 5.4M in 2024) and falling GMV ($932M to $721M). Not a growth story.



### Step 2 — Profitability Trajectory

Examines four margin layers: Gross, Operating, Adjusted EBITDA, and Net.
The key question is not whether a company is profitable yet — it is whether the gap between gross margin and operating margin is closing. A shrinking gap means operating costs are being absorbed as revenue scales. That is operating leverage.

In [12]:
print("--- Margin Summary (%) ---\n")

margin_cols  = ['Company', 'Year', 'Gross_Margin_Pct', 'Operating_Margin_Pct',
                'EBITDA_Margin_Pct', 'Net_Margin_Pct']
margin_table = master_df[margin_cols].copy()

for col in ['Gross_Margin_Pct', 'Operating_Margin_Pct', 'EBITDA_Margin_Pct', 'Net_Margin_Pct']:
    margin_table[col] = margin_table[col].apply(lambda x: f"{x:+.1f}%" if pd.notna(x) else "—")

print(margin_table.to_string(index=False))

--- Margin Summary (%) ---

      Company  Year Gross_Margin_Pct Operating_Margin_Pct EBITDA_Margin_Pct Net_Margin_Pct
Delivery Hero  2022           +26.0%                -9.9%             -5.4%         -35.1%
Delivery Hero  2023           +29.9%                -1.8%             +2.6%         -23.1%
Delivery Hero  2024           +27.1%                +3.6%             +5.6%          -7.2%
     DoorDash  2022           +45.5%               -15.7%             +5.5%         -20.7%
     DoorDash  2023           +46.9%                -6.7%            +13.8%          -6.5%
     DoorDash  2024           +48.3%                -0.4%            +17.7%          +1.1%
        Jumia  2022           +58.1%               -94.0%            -89.6%        -117.2%
        Jumia  2023           +57.5%               -38.8%            -31.2%         -55.9%
        Jumia  2024           +59.4%               -38.4%            -30.6%         -59.2%


In [13]:
print("--- Operating Leverage Gap ---")
print("    Gross Margin minus Operating Margin — smaller gap means better cost absorption\n")

for company in ['DoorDash', 'Delivery Hero', 'Jumia']:
    subset = master_df[master_df['Company'] == company].sort_values('Year')
    print(f"{company}")
    for _, row in subset.iterrows():
        gap = row['Gross_Margin_Pct'] - row['Operating_Margin_Pct']
        print(f"  {int(row['Year'])}: Gross {row['Gross_Margin_Pct']:+.1f}%  |  "
              f"Operating {row['Operating_Margin_Pct']:+.1f}%  |  Gap = {gap:.1f}pts")
    print()

--- Operating Leverage Gap ---
    Gross Margin minus Operating Margin — smaller gap means better cost absorption

DoorDash
  2022: Gross +45.5%  |  Operating -15.7%  |  Gap = 61.2pts
  2023: Gross +46.9%  |  Operating -6.7%  |  Gap = 53.6pts
  2024: Gross +48.3%  |  Operating -0.4%  |  Gap = 48.7pts

Delivery Hero
  2022: Gross +26.0%  |  Operating -9.9%  |  Gap = 35.9pts
  2023: Gross +29.9%  |  Operating -1.8%  |  Gap = 31.7pts
  2024: Gross +27.1%  |  Operating +3.6%  |  Gap = 23.5pts

Jumia
  2022: Gross +58.1%  |  Operating -94.0%  |  Gap = 152.1pts
  2023: Gross +57.5%  |  Operating -38.8%  |  Gap = 96.3pts
  2024: Gross +59.4%  |  Operating -38.4%  |  Gap = 97.8pts



In [14]:
print("--- Adj. EBITDA Margin Trajectory ---\n")

for company in ['DoorDash', 'Delivery Hero', 'Jumia']:
    subset    = master_df[master_df['Company'] == company].sort_values('Year')
    margins   = subset['EBITDA_Margin_Pct'].tolist()
    start, end = margins[0], margins[-1]
    change    = end - start
    direction = "▲" if change > 0 else "▼"
    print(f"{company}: {start:+.1f}% (2022)  -->  {end:+.1f}% (2024)  "
          f"{direction} {abs(change):.1f}pts")

--- Adj. EBITDA Margin Trajectory ---

DoorDash: +5.5% (2022)  -->  +17.7% (2024)  ▲ 12.2pts
Delivery Hero: -5.4% (2022)  -->  +5.6% (2024)  ▲ 11.0pts
Jumia: -89.6% (2022)  -->  -30.6% (2024)  ▲ 59.0pts


In [15]:
print("--- Analytical Observations ---\n")

observations = [
    "DoorDash — Gross Margin expanded from 46% to 48%, but the real story is "
    "the operating leverage gap closing from 62pts in 2022 to 49pts in 2024. "
    "EBITDA Margin went from +6% to +18% in two years. "
    "The fixed cost base is being absorbed as GMV scales.",

    "Delivery Hero — Gross Margin is the lowest of the three (26-27%) and barely moving. "
    "Positive Operating Margin in 2024 (+4%) is encouraging, but Net Income "
    "remains deeply negative (-$957M) due to interest and restructuring costs.",

    "Jumia — Highest Gross Margin (~59%) but Operating Margin of -38% to -94%. "
    "The 55pt+ gap between gross and operating margin is not closing meaningfully. "
    "Fulfillment and logistics costs in underdeveloped African markets "
    "consume everything the gross profit generates. "
    "High gross margin here is a structural trap, not a strength.",
]

for i, obs in enumerate(observations, 1):
    print(f"{i}. {obs}\n")

--- Analytical Observations ---

1. DoorDash — Gross Margin expanded from 46% to 48%, but the real story is the operating leverage gap closing from 62pts in 2022 to 49pts in 2024. EBITDA Margin went from +6% to +18% in two years. The fixed cost base is being absorbed as GMV scales.

2. Delivery Hero — Gross Margin is the lowest of the three (26-27%) and barely moving. Positive Operating Margin in 2024 (+4%) is encouraging, but Net Income remains deeply negative (-$957M) due to interest and restructuring costs.

3. Jumia — Highest Gross Margin (~59%) but Operating Margin of -38% to -94%. The 55pt+ gap between gross and operating margin is not closing meaningfully. Fulfillment and logistics costs in underdeveloped African markets consume everything the gross profit generates. High gross margin here is a structural trap, not a strength.



### Step 3 — Cash Position

Examines Operating Cash Flow, cash balance trend, and cash runway.
The question here: can the business fund itself, or does it depend on external capital to survive?

In [16]:
print("--- Cash & Operating Cash Flow (USD Millions) ---\n")

cash_cols  = ['Company', 'Year', 'Cash_USD', 'Operating_CF_USD', 'Cash_Runway_Months']
cash_table = master_df[cash_cols].copy()

cash_table['Cash_USD']           = cash_table['Cash_USD'].apply(lambda x: f"${x:,.0f}M")
cash_table['Operating_CF_USD']   = cash_table['Operating_CF_USD'].apply(
    lambda x: f"${x:+,.0f}M" if pd.notna(x) else "—"
)
cash_table['Cash_Runway_Months'] = cash_table['Cash_Runway_Months'].apply(
    lambda x: f"{x:.0f} months" if pd.notna(x) else "N/A (cash-generative)"
)

print(cash_table.to_string(index=False))

--- Cash & Operating Cash Flow (USD Millions) ---

      Company  Year Cash_USD Operating_CF_USD    Cash_Runway_Months
Delivery Hero  2022  $2,546M           $-725M             42 months
Delivery Hero  2023  $1,796M            $-21M           1021 months
Delivery Hero  2024  $4,132M           $+693M N/A (cash-generative)
     DoorDash  2022  $1,977M           $+367M N/A (cash-generative)
     DoorDash  2023  $2,656M         $+1,673M N/A (cash-generative)
     DoorDash  2024  $4,019M         $+2,132M N/A (cash-generative)
        Jumia  2022     $72M           $-240M              4 months
        Jumia  2023     $36M            $-73M              6 months
        Jumia  2024     $55M            $-57M             12 months


In [17]:
print("--- Cash Flow Status by Year ---\n")

for company in ['DoorDash', 'Delivery Hero', 'Jumia']:
    subset = master_df[master_df['Company'] == company].sort_values('Year')
    print(f"{company}")
    for _, row in subset.iterrows():
        cf     = row['Operating_CF_USD']
        status = "GENERATING" if cf >= 0 else "BURNING"
        print(f"  {int(row['Year'])}: ${cf:+,.0f}M  —  {status}")
    print()

--- Cash Flow Status by Year ---

DoorDash
  2022: $+367M  —  GENERATING
  2023: $+1,673M  —  GENERATING
  2024: $+2,132M  —  GENERATING

Delivery Hero
  2022: $-725M  —  BURNING
  2023: $-21M  —  BURNING
  2024: $+693M  —  GENERATING

Jumia
  2022: $-240M  —  BURNING
  2023: $-73M  —  BURNING
  2024: $-57M  —  BURNING



In [18]:
print("--- Net Cash Movement (2022 to 2024) ---\n")

for company in ['DoorDash', 'Delivery Hero', 'Jumia']:
    subset  = master_df[master_df['Company'] == company].sort_values('Year')
    cash_22 = subset[subset['Year'] == 2022]['Cash_USD'].values[0]
    cash_24 = subset[subset['Year'] == 2024]['Cash_USD'].values[0]
    change  = cash_24 - cash_22
    direction = "▲" if change > 0 else "▼"
    print(f"{company}: ${cash_22:,.0f}M (2022)  -->  ${cash_24:,.0f}M (2024)  "
          f"{direction} ${abs(change):,.0f}M")

--- Net Cash Movement (2022 to 2024) ---

DoorDash: $1,977M (2022)  -->  $4,019M (2024)  ▲ $2,042M
Delivery Hero: $2,546M (2022)  -->  $4,132M (2024)  ▲ $1,586M
Jumia: $72M (2022)  -->  $55M (2024)  ▼ $16M


In [19]:
print("--- Analytical Observations ---\n")

observations = [
    "DoorDash — Cash-generative in all three years. "
    "Operating Cash Flow grew from $367M to $2.13B — a 5.8x increase in two years. "
    "Cash balance grew from $1.98B to $4.02B. "
    "This company does not need external capital to operate or grow.",

    "Delivery Hero — Turned cash-generative in 2024 (+$693M OCF). "
    "Cash balance dropped from $2.55B to $1.80B between 2022 and 2023 "
    "before recovering to $4.13B in 2024 — partly supported by the Talabat IPO. "
    "The recovery is real but not purely operational.",

    "Jumia — 4 months of cash runway in 2022 signals near-insolvency. "
    "Runway improved to 12 months in 2024 only because burn was cut, not because "
    "the business grew. Cash balance fell from $72M to $35M in 2023 before partially "
    "recovering to $55M in 2024. With declining revenue and no path to positive OCF, "
    "Jumia remains structurally dependent on external funding.",
]

for i, obs in enumerate(observations, 1):
    print(f"{i}. {obs}\n")

--- Analytical Observations ---

1. DoorDash — Cash-generative in all three years. Operating Cash Flow grew from $367M to $2.13B — a 5.8x increase in two years. Cash balance grew from $1.98B to $4.02B. This company does not need external capital to operate or grow.

2. Delivery Hero — Turned cash-generative in 2024 (+$693M OCF). Cash balance dropped from $2.55B to $1.80B between 2022 and 2023 before recovering to $4.13B in 2024 — partly supported by the Talabat IPO. The recovery is real but not purely operational.

3. Jumia — 4 months of cash runway in 2022 signals near-insolvency. Runway improved to 12 months in 2024 only because burn was cut, not because the business grew. Cash balance fell from $72M to $35M in 2023 before partially recovering to $55M in 2024. With declining revenue and no path to positive OCF, Jumia remains structurally dependent on external funding.



### Step 4 — Investment Scorecard & Conclusion

Each company ranked 1 (best) to 3 (worst) across 8 metrics.
Lower total score = stronger investment profile.

In [20]:
print("--- Investment Scorecard (1 = Best, 3 = Worst) ---\n")

scorecard = {
    'Metric': [
        'Revenue Growth (2-Yr CAGR)',
        'Gross Margin % (2024)',
        'Operating Margin Direction',
        'EBITDA Margin (2024)',
        'Net Income (2024)',
        'Operating Cash Flow (2024)',
        'Cash Runway Risk',
        'Cash Balance Growth',
    ],
    'DoorDash':      [1, 2, 2, 1, 1, 1, 1, 1],
    'Delivery Hero': [2, 3, 1, 2, 3, 2, 2, 2],
    'Jumia':         [3, 1, 3, 3, 2, 3, 3, 3],
    'Notes': [
        'DASH 27.6% | DHER 21.5% | JMIA -9.2%',
        'JMIA 59% but operationally hollow | DASH 48% | DHER 27%',
        'DHER gap closing fastest | DASH near breakeven | JMIA stagnant',
        'DASH +17.7% | DHER +5.6% | JMIA -30.6%',
        'DASH only profitable company (+$123M)',
        'DASH $2.13B | DHER $693M | JMIA -$57M',
        'DASH no risk | DHER improving | JMIA structurally fragile',
        'DASH +$2.04B | DHER +$1.59B (partly Talabat IPO) | JMIA -$16M',
    ]
}

sc_df = pd.DataFrame(scorecard)
print(sc_df.to_string(index=False))

--- Investment Scorecard (1 = Best, 3 = Worst) ---

                    Metric  DoorDash  Delivery Hero  Jumia                                                          Notes
Revenue Growth (2-Yr CAGR)         1              2      3                           DASH 27.6% | DHER 21.5% | JMIA -9.2%
     Gross Margin % (2024)         2              3      1        JMIA 59% but operationally hollow | DASH 48% | DHER 27%
Operating Margin Direction         2              1      3 DHER gap closing fastest | DASH near breakeven | JMIA stagnant
      EBITDA Margin (2024)         1              2      3                         DASH +17.7% | DHER +5.6% | JMIA -30.6%
         Net Income (2024)         1              3      2                          DASH only profitable company (+$123M)
Operating Cash Flow (2024)         1              2      3                          DASH $2.13B | DHER $693M | JMIA -$57M
          Cash Runway Risk         1              2      3      DASH no risk | DHER improving 

In [21]:
print("--- Total Score (lower is better) ---\n")

for company in ['DoorDash', 'Delivery Hero', 'Jumia']:
    total = sum(scorecard[company])
    print(f"  {company}: {total} / 24")

--- Total Score (lower is better) ---

  DoorDash: 10 / 24
  Delivery Hero: 17 / 24
  Jumia: 21 / 24


In [22]:
print("=" * 60)
print("INVESTMENT CONCLUSION")
print("=" * 60)

conclusion = """
Question: Which company represents the best investment opportunity
          in Quick Commerce?

Answer: DoorDash (DASH)

Rationale:

DoorDash is the only company in this analysis to achieve all three
milestones of investment-grade maturity simultaneously:

  1. PROFITABILITY — Positive Net Income of +$123M in 2024.
     First among the three to cross this threshold.

  2. CASH GENERATION — Operating Cash Flow of $2.13B in 2024,
     up from $367M in 2022. A 5.8x increase driven entirely
     by core operations — no asset sales, no external funding.

  3. SCALE WITH IMPROVING MARGINS — GMV scaled from $53.4B to
     $80.2B while EBITDA Margin expanded from +5.5% to +17.7%.
     This is operating leverage working at scale.

Why not Delivery Hero?
  A turnaround story, not a maturity story. OCF turned positive
  in 2024, but Net Income remains -$957M due to structural costs
  below the operating line. The cash recovery was partly driven
  by the Talabat IPO, not purely operations.

Why not Jumia?
  A survival story. Revenue declined two consecutive years.
  Active customers fell from 7.3M to 5.4M. Cash runway was
  4 months in 2022. No visible path to positive OCF.
  The 59% gross margin is structurally hollow — consumed by
  fulfillment costs in underdeveloped African markets.

Caveat:
  Group-level public financials used as proxy throughout.
  Segment-level data not available in public filings.
  Delivery Hero FX conversion uses average annual EUR/USD rates.
  Data sourced from yfinance and annual reports (2022-2024).
"""

print(conclusion)

INVESTMENT CONCLUSION

Question: Which company represents the best investment opportunity
          in Quick Commerce?

Answer: DoorDash (DASH)

Rationale:

DoorDash is the only company in this analysis to achieve all three
milestones of investment-grade maturity simultaneously:

  1. PROFITABILITY — Positive Net Income of +$123M in 2024.
     First among the three to cross this threshold.

  2. CASH GENERATION — Operating Cash Flow of $2.13B in 2024,
     up from $367M in 2022. A 5.8x increase driven entirely
     by core operations — no asset sales, no external funding.

  3. SCALE WITH IMPROVING MARGINS — GMV scaled from $53.4B to
     $80.2B while EBITDA Margin expanded from +5.5% to +17.7%.
     This is operating leverage working at scale.

Why not Delivery Hero?
  A turnaround story, not a maturity story. OCF turned positive
  in 2024, but Net Income remains -$957M due to structural costs
  below the operating line. The cash recovery was partly driven
  by the Talabat IPO, not pu